# Mass–spring equilibrium: data reduction

For each sample (unstretched length, 100 g, 201 g, 301 g):
1. read the file with `np.loadtxt`,
2. record the time, distance and distance-error arrays,
3. average the distances to get the equilibrium position,
4. propagate the error in quadrature: $\sigma = \sqrt{\sigma_{std}^2 + \bar{\delta}^2}$, where $\bar{\delta}$ is the mean sensor error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from springmass import count_header_lines

## 1–2. Read the files and record the data

The files start with 2 or 3 title lines before the `Time (s)` header, so `count_header_lines` finds how many rows to skip.

In [ ]:
data_dir = Path("data/equilibrium")
samples = ["unstretchedlength", "100gram", "201gram", "301gram"]

data = {}
for name in samples:
    path = data_dir / name / "positionvssample.txt"
    skip = count_header_lines(path)
    time, dist, dist_err = np.loadtxt(path, delimiter="\t", skiprows=skip, unpack=True)
    data[name] = {"time": time, "dist": dist, "dist_err": dist_err}
    print(f"{name:>18}: skipped {skip} header lines, {len(dist)} points")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, d in data.items():
    ax.plot(d["time"], d["dist"], label=name)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance (cm)")
ax.legend()
plt.show()

## 3–4. Mean equilibrium position and uncertainty

In [ ]:
means, sigmas = {}, {}
print(f"{'sample':>18} {'n':>4} {'mean':>8} {'std':>7} {'err':>7} {'sigma':>7}  (cm)")
for name, d in data.items():
    dist, dist_err = d["dist"], d["dist_err"]
    mean = np.mean(dist)
    std = np.std(dist, ddof=1)
    err_mean = np.mean(dist_err)
    sigma = np.sqrt(std**2 + err_mean**2)
    means[name], sigmas[name] = mean, sigma
    print(f"{name:>18} {len(dist):>4} {mean:8.3f} {std:7.3f} {err_mean:7.3f} {sigma:7.3f}")

In [ ]:
for name in samples:
    print(f"{name:>18}: x_eq = {means[name]:.2f} ± {sigmas[name]:.2f} cm")